# Phase 2 — Spectrum codec (architecture & forward pass)

ConvNeXt-style 1D encoder + lookup-free quantizer (LFQ) + decoder; fixed **8704** grid and **273** latent tokens.

**Kernel:** choose **Python 3.x (.venv)** from `final-project/.venv` (not a random system Python). Re-run `pip install -e .` from the repo root if imports fail.

The first code cell adds `src/` to `sys.path` whether Jupyter’s working directory is the repo root or `notebooks/`. It does **not** use raw `%matplotlib inline` (that breaks outside IPython).


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

_cwd = Path.cwd().resolve()
if (_cwd / "src" / "desifm").is_dir():
    REPO = _cwd
elif (_cwd.parent / "src" / "desifm").is_dir():
    REPO = _cwd.parent
else:
    raise FileNotFoundError(f"Could not find repo root (src/desifm). cwd={_cwd}")
sys.path.insert(0, str(REPO / "src"))

try:
    from IPython import get_ipython

    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("matplotlib", "inline")
except Exception:
    pass

plt.rcParams["figure.figsize"] = (11, 4)

from desifm.constants import GRID_SIZE, N_LATENT_TOKENS
from desifm.tokenization.spectrum_codec import SpectrumCodec
from desifm.training.codec_input import denormalize_spectrum_output, normalize_spectrum_input

print("GRID_SIZE =", GRID_SIZE, "  N_LATENT_TOKENS =", N_LATENT_TOKENS)


## Model footprint

Parameter count and default channel widths (see `SpectrumCodec` in `src/desifm/tokenization/spectrum_codec.py`).


In [ ]:
model = SpectrumCodec()
n_params = sum(p.numel() for p in model.parameters())
print(f"SpectrumCodec parameters: {n_params:,}")
print(model)


## Synthetic forward pass

Random `(B, 2, L)` arcsinh-normalized tensor (flux + istd channels). `forward` returns reconstruction in **arcsinh** space, Huber `recon_loss` on flux vs input, and LFQ `q_loss`.


In [ ]:
torch.manual_seed(0)
B, L = 4, 4096
flux = torch.rand(B, L) * 0.5 + 0.5
ivar = torch.ones(B, L) * 10.0
mask = torch.zeros(B, L, dtype=torch.bool)
x, denorm = normalize_spectrum_input(flux, ivar, mask)
with torch.no_grad():
    out = model(x, denorm, mask=mask)

print("x shape:", tuple(x.shape), " denorm:", tuple(denorm.shape))
print("indices shape:", tuple(out["indices"].shape), "(B, n_tokens)")
for k in ("recon_loss", "q_loss", "loss"):
    print(f"  {k}: {float(out[k].item()):.6f}")

# Physical-space round-trip of the *input* (not quantized recon)
phys = denormalize_spectrum_output(x, denorm)
print("physical flux shape:", tuple(phys.shape), " max |flux_err|:", float((phys[:, 0] - flux).abs().max()))


## Loss breakdown (bar)

Single minibatch — useful for smoke; training curves live in W&B / `metrics.jsonl`.


In [ ]:
fig, ax = plt.subplots()
keys = ["recon_loss", "q_loss"]
vals = [float(out[k].item()) for k in keys]
ax.bar(keys, vals, color=["steelblue", "coral"])
ax.set_ylabel("loss (arcsinh flux / LFQ)")
ax.set_title("Codec forward — loss components (one batch)")
plt.show()


## Encode → decode token path

Indices are packed LFQ binary codes per latent position.


In [ ]:
with torch.no_grad():
    idx, _ = model.encode(x, denorm)
    recon_from_idx = model.decode(idx, denorm, to_physical=True)
L_in = flux.shape[1]
# Decoder runs on the codec's fixed GRID_SIZE; resample flux channel back to input length for comparison.
recon_flux_grid = recon_from_idx[:, 0:1, :]
recon_at_in = torch.nn.functional.interpolate(
    recon_flux_grid, size=L_in, mode="linear", align_corners=False
).squeeze(1)
print("token index min/max:", int(idx.min()), int(idx.max()))
print("decode physical shape (B, 2, GRID_SIZE):", tuple(recon_from_idx.shape))
print("max |flux decode - GT| @ input length:", float((recon_at_in - flux).abs().max()))


## Train on real data

From repo root (or NERSC), see `scripts/train_codec.py` and `docs/NERSC_INTERACTIVE.md`.

```bash
python scripts/train_codec.py --manifest /path/to/manifest.jsonl --run-name codec_v3 --wandb-mode online
```
